# VoiceHub inference

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadirnar/voicehub/blob/main/notebooks/inference.ipynb)

Edit one configuration cell, enable one task, and run top to bottom. Model downloads and audio execution are off by default.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("voicehub") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "voicehub @ git+https://github.com/kadirnar/voicehub.git@main",
    ])

## 1. Configure

Use the same text, seed, checkpoint revision, and device for every comparison.

In [ ]:
from pathlib import Path

RUN_TTS = False
RUN_TTS_OPTIMIZATION = False
RUN_ASR = False
RUN_VAD = False

DEVICE = "cuda"
AUDIO_PATH = Path("data/speech.wav")
OUTPUT_DIR = Path("artifacts/inference")

TTS_MODEL_TYPE = "parlertts"
TTS_CHECKPOINT = "parler-tts/parler-tts-mini-v1"
ASR_MODEL_TYPE = "asr_qwen3"
ASR_CHECKPOINT = "Qwen/Qwen3-ASR-0.6B"
VAD_MODEL_TYPE = "vad_silero"

TTS_SAMPLES = (
    "VoiceHub keeps speech experiments easy to inspect and repeat. This long sample checks pronunciation, pacing, pauses, and consistency across several complete sentences. The same prompt and seed can then be reused to compare eager inference with each supported optimization on the selected machine. Listen for stable volume, clear endings, and natural transitions throughout the complete recording.",
    "A reliable voice system should remain clear through complete paragraphs, not only short greetings. This sample includes varied sentence lengths and natural punctuation so listeners can evaluate timing, transitions, intelligibility, and sustained audio quality for more than ten seconds. It also gives the model enough context to reveal changes in tone, rhythm, or emphasis near the end.",
    "Before reporting a faster result, warm up the model, synchronize the accelerator, and measure the exact same request. Record latency, generated duration, and peak memory, then listen to both files to confirm that performance changes did not reduce speech quality. Repeat the comparison several times and preserve the generated audio with the exact configuration used for each measured run.",
)
assert min(len(text.split()) for text in TTS_SAMPLES) >= 55

## 2. Check the registry

In [ ]:
from collections import Counter

from voicehub import list_model_specs

catalog = list_model_specs(task=None)
task_counts = Counter(spec.task.value for spec in catalog)
print(dict(task_counts))
for model_type in (TTS_MODEL_TYPE, ASR_MODEL_TYPE, VAD_MODEL_TYPE):
    spec = next(item for item in catalog if item.model_type == model_type)
    print(spec.model_type, spec.default_model_path, spec.training.support.value)

## 3. Generate speech

The word count is only a starting point. The cell raises if the returned waveform is shorter than ten seconds.

In [ ]:
if RUN_TTS:
    from voicehub import AutoModelForTextToSpeech, TTSGenerationConfig

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    tts_model = AutoModelForTextToSpeech.from_pretrained(
        TTS_CHECKPOINT,
        model_type=TTS_MODEL_TYPE,
        device=DEVICE,
        lazy_load=True,
    )
    tts_output = tts_model.generate(
        TTS_SAMPLES[0],
        description="A clear speaker talks at a steady, natural pace.",
        generation_config=TTSGenerationConfig(
            seed=42,
            output_file=OUTPUT_DIR / "tts-eager.wav",
        ),
    )
    sample_count = tts_output.audio.shape[-1] if hasattr(tts_output.audio, "shape") else len(tts_output.audio)
    tts_duration = sample_count / tts_output.sample_rate
    if tts_duration < 10.0:
        raise RuntimeError(f"Expected at least 10 seconds, got {tts_duration:.2f}")
    print(tts_output.file_path, f"{tts_duration:.2f}s")

## 4. Apply quality-preserving TTS optimization

Run eager inference first. The optimization manifest tells you what actually changed; measure speed and peak memory on your hardware.

In [ ]:
if RUN_TTS and RUN_TTS_OPTIMIZATION:
    from voicehub import TTSGenerationConfig
    from voicehub.optimization import TTSOptimizationConfig

    optimization_result = tts_model.optimize(
        TTSOptimizationConfig(
            attn_implementation="auto",
            kernel_backend="auto",
            compile="auto",
        )
    )
    print(optimization_result.manifest())
    optimized_output = tts_model.generate(
        TTS_SAMPLES[0],
        description="A clear speaker talks at a steady, natural pace.",
        generation_config=TTSGenerationConfig(
            seed=42,
            output_file=OUTPUT_DIR / "tts-optimized.wav",
        ),
    )
    optimized_samples = optimized_output.audio.shape[-1] if hasattr(optimized_output.audio, "shape") else len(optimized_output.audio)
    optimized_duration = optimized_samples / optimized_output.sample_rate
    if optimized_duration < 10.0:
        raise RuntimeError(f"Expected at least 10 seconds, got {optimized_duration:.2f}")

## 5. Run ASR or VAD

In [ ]:
if RUN_ASR:
    from voicehub import AutoModelForSpeechRecognition

    if not AUDIO_PATH.is_file():
        raise FileNotFoundError(AUDIO_PATH)
    asr_model = AutoModelForSpeechRecognition.from_pretrained(
        ASR_CHECKPOINT,
        model_type=ASR_MODEL_TYPE,
        device=DEVICE,
    )
    asr_output = asr_model.transcribe(AUDIO_PATH, language="English")
    print(asr_output.text)

In [ ]:
if RUN_VAD:
    from voicehub import AutoModelForVoiceActivityDetection

    if not AUDIO_PATH.is_file():
        raise FileNotFoundError(AUDIO_PATH)
    vad_model = AutoModelForVoiceActivityDetection.from_pretrained(
        model_type=VAD_MODEL_TYPE,
        device="cpu",
    )
    vad_output = vad_model.detect(AUDIO_PATH, threshold=0.55)
    for segment in vad_output.segments:
        print(segment.start, segment.end, segment.score)

## Next

Use the [inference guide](https://kadirnar.github.io/voicehub/guides/inference/), [optimization guide](https://kadirnar.github.io/voicehub/guides/tts-optimization/), [speech-recognition guide](https://kadirnar.github.io/voicehub/guides/speech-recognition/), [VAD guide](https://kadirnar.github.io/voicehub/guides/voice-activity-detection/), and [recorded benchmarks](https://kadirnar.github.io/voicehub/guides/rtx-5090-tts-benchmarks/) for model-specific details.